In [ ]:
# Nitsa Saint Fort
# 10/14/25
# CAP 4770
# Assignment 3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.svm import SVC
from sklearn.metrics import f1_score

In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
path = 'drive/My Drive/Activity_data.csv'

In [ ]:
df=pd.read_csv(path)

Part 1

In [ ]:
df.head()

,Time,frontal_axis_Acceleration,vertical_axis+_Acceleration,lateral_axis_Acceleration,Antenna_ID_Sensor,RSSI,Phase,Frequency,Activity
0,0.00,-0.232160,0.215880,-1.18820,2,-48.5,2.74430,920.75,3
1,0.25,-0.056282,0.043636,-1.14260,2,-50.0,2.36540,921.75,3
2,0.50,0.143050,0.296270,-1.33640,3,-59.5,0.20709,922.25,3
3,0.75,0.318930,0.284780,-0.92593,2,-51.0,0.41571,923.25,3
4,1.05,0.412730,0.135500,-1.02860,2,-44.5,4.45470,920.25,3


In [ ]:
D = df.iloc[:,1:-1].to_numpy()
y = df.iloc[:,-1].to_numpy()

In [ ]:
# Splitting the data into train and test
X_train, X_Test, y_train, y_test = train_test_split (D,y,test_size=0.2)

Part 2

In [ ]:
# Bayes classifier from scratch
# Build Bayes classifier
class BayesClassifier:
    def __init__(self):
        self.classes = None
        self.priors = None
        self.mean = None
        self.variance = None

    def fit(self, X, y):
        """
        Train the Bayes classifier.

        Parameters:
        X: np.ndarray
            Feature matrix of shape (n_samples, n_features)
        y: np.ndarray
            Class labels of shape (n_samples,)
        """
        self.classes = np.unique(y)
        n_features = X.shape[1]
        n_classes = len(self.classes)

        # Initialize arrays to store class-wise statistics
        self.mean = np.zeros((n_classes, n_features))
        self.variance = np.zeros((n_classes, n_features))
        self.priors = np.zeros(n_classes)

        for idx, cls in enumerate(self.classes):
            X_c = X[y == cls]
            self.mean[idx, :] = X_c.mean(axis=0)
            self.variance[idx, :] = X_c.var(axis=0)
            self.priors[idx] = X_c.shape[0] / float(X.shape[0])

    def _calculate_likelihood(self, mean, var, x):
        """
        Calculate the Gaussian likelihood of the data.
        """
        eps = 1e-6  # Add small epsilon to variance to avoid division by zero
        coeff = 1 / np.sqrt(2 * np.pi * var + eps)
        exponent = -((x - mean) ** 2) / (2 * (var + eps))
        return coeff * np.exp(exponent)

    def _calculate_posterior(self, X):
        """
        Calculate posterior probability for each class.
        """
        posteriors = []
        for idx, cls in enumerate(self.classes):
            prior = np.log(self.priors[idx])
            likelihood = np.sum(np.log(self._calculate_likelihood(self.mean[idx, :], self.variance[idx, :], X)), axis=1)
            posteriors.append(prior + likelihood)
        return np.array(posteriors).T

    def predict(self, X):
        """
        Predict class labels for the input data.
        """
        posteriors = self._calculate_posterior(X)
        return self.classes[np.argmax(posteriors, axis=1)]

In [ ]:
# Function for Linear SVM
lin_svc = LinearSVC(penalty = 'l2')
lin_svc.fit(X_train, y_train)
print(lin_svc)

LinearSVC()


In [ ]:
# Function for Kernel SVM
kernel_svc = SVC(kernel = 'rbf')
kernel_svc.fit(X_train, y_train)
print(kernel_svc)

SVC()


In [ ]:
# k_values
k_values = [1, 3, 5, 7]

In [ ]:
# KNN
for idx, k in enumerate(k_values, start=2):
    # Train KNN model
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)

    # Predict for acceleration
    prediction = knn.predict(df)

    # Find the k nearest neighbors
    distances, indices = knn.kneighbors(df)

    print(f"\n--- K = {k} ---")
    print(f"Looking at {k} nearest neighbor(s):")

    for i, neighbor_idx in enumerate(indices[0]):
        neighbor_data = X_train[neighbor_idx]
        neighbor_label = "PASSED" if y_train[neighbor_idx] == 1 else "FAILED"
        print(f"  Neighbor {i+1}: Hours={neighbor_data[0]}, Score={neighbor_data[1]}, Status={neighbor_label}")

    result = "PASS ✓" if prediction[0] == 1 else "FAIL ✗"
    print(f"Prediction: {result}")




    # Visualize
    plt.subplot(2, 2, idx)
    plt.scatter(df[:, 0], df[:, 1], c='red', s=100,
                label='Failed', alpha=0.5, edgecolors='black')
    plt.scatter(df[:, 0], df[:, 1], c='blue', s=100,
                label='Passed', alpha=0.5, edgecolors='black')

    # Highlight the k nearest neighbors
    for neighbor_idx in indices[0]:
        neighbor = X_train[neighbor_idx]
        plt.scatter(neighbor[0], neighbor[1], s=300,
                   facecolors='none', edgecolors='green', linewidth=3)
        plt.plot([df[0][0], neighbor[0]],
                [df[0][1], neighbor[1]],
                'g--', alpha=0.5, linewidth=2)

    # Plot acceleration
    color = 'blue' if prediction[0] == 1 else 'red'
    plt.scatter(df[0][0], df[0][1], c=color, s=200,
               marker='*', edgecolors='black', linewidth=2, label='Activity', zorder=5)

    plt.xlabel('frontal-axis acceleration', fontsize=12)
    plt.ylabel('vertical axis+ acceleration', fontsize=12)
    plt.title(f'K = {k}: Prediction = {result}', fontsize=14, fontweight='bold')
    plt.legend(fontsize=9)
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('knn_visualization.png', dpi=100, bbox_inches='tight')
print("\n✓ Visualization saved as 'knn_visualization.png'")
plt.show()

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


ValueError: X has 9 features, but KNeighborsClassifier is expecting 7 features as input.

In [ ]:
# Declaring the function for y_true and y_score
y_true = y_test
y_score = lin_svc.decision_function(X_Test)

In [ ]:
# Naive Bayes Classifier
NB= GaussianNB()
NB.fit(X_train, y_train)
y_pred_NB = NB.predict(X_Test)
AP = average_precision_score(y_true, y_score, df)
print(accuracy_score(y_test, y_pred_NB))
print(average_precision_score(df))

TypeError: too many positional arguments

In [ ]:
# LDA classifier
LDA = LinearDiscriminantAnalysis()
LDA.fit(X_train,y_train)
y_pred = LDA.predict(X_Test)
print(accuracy_score(y_test,y_pred))

0.909889524823639


In [ ]:
# F1-Score
f1_score_value = f1_score(y_test, y_pred_NB, average='weighted')
print(f1_score_value)

0.93732383259449


In [ ]:
# Precision Score
precision_score_value = precision_score(y_test,y_pred_NB, average='weighted')
print(precision_score_value)

0.9404950943448505


In [ ]:
# Recall Score
recall_score_value = recall_score(y_test,y_pred_NB, average='weighted')
print(recall_score_value)

0.9415013975775323


Part 3

In [ ]:
data1 = df.select_dtypes(include=["number"])
data1 = data1.dropna()
data1 = data1.to_numpy()
print(data1)
print("Array dtype:", data1.dtype)
print("Any object cells:", (data1.dtype == object))

[[ 0.0000e+00 -2.3216e-01  2.1588e-01 ...  2.7443e+00  9.2075e+02
   3.0000e+00]
 [ 2.5000e-01 -5.6282e-02  4.3636e-02 ...  2.3654e+00  9.2175e+02
   3.0000e+00]
 [ 5.0000e-01  1.4305e-01  2.9627e-01 ...  2.0709e-01  9.2225e+02
   3.0000e+00]
 ...
 [ 4.8550e+02  3.3066e-01  9.5081e-01 ...  5.0054e+00  9.2475e+02
   1.0000e+00]
 [ 4.8825e+02  1.1960e-01  8.2449e-01 ...  4.9087e+00  9.2575e+02
   4.0000e+00]
 [ 4.8875e+02  7.2698e-02  1.3183e+00 ...  1.0370e+00  9.2275e+02
   4.0000e+00]]
Array dtype: float64
Any object cells: False


In [ ]:
#1) PCA with Dimensionality Reduction
pca = PCA(n_components = 0.96)
pca.fit(data1)
data1_red = pca.transform(data1)
print(data1_red.shape)

(75128, 1)


Part 4

In [ ]:
class BayesClassifier:
    def __init__(self):
        self.classes = None
        self.priors = None
        self.mean = None
        self.variance = None

    def fit(self, X, y):
        """
        Train the Bayes classifier.

        Parameters:
        X: np.ndarray
            Feature matrix of shape (n_samples, n_features)
        y: np.ndarray
            Class labels of shape (n_samples,)
        """
        self.classes = np.unique(y)
        n_features = X.shape[1]
        n_classes = len(self.classes)

        # Initialize arrays to store class-wise statistics
        self.mean = np.zeros((n_classes, n_features))
        self.variance = np.zeros((n_classes, n_features))
        self.priors = np.zeros(n_classes)

        for idx, cls in enumerate(self.classes):
            X_c = X[y == cls]
            self.mean[idx, :] = X_c.mean(axis=0)
            self.variance[idx, :] = X_c.var(axis=0)
            self.priors[idx] = X_c.shape[0] / float(X.shape[0])

    def _calculate_likelihood(self, mean, var, x):
        """
        Calculate the Gaussian likelihood of the data.
        """
        eps = 1e-6  # Add small epsilon to variance to avoid division by zero
        coeff = 1 / np.sqrt(2 * np.pi * var + eps)
        exponent = -((x - mean) ** 2) / (2 * (var + eps))
        return coeff * np.exp(exponent)

    def _calculate_posterior(self, X):
        """
        Calculate posterior probability for each class.
        """
        posteriors = []
        for idx, cls in enumerate(self.classes):
            prior = np.log(self.priors[idx])
            likelihood = np.sum(np.log(self._calculate_likelihood(self.mean[idx, :], self.variance[idx, :], X)), axis=1)
            posteriors.append(prior + likelihood)
        return np.array(posteriors).T

    def predict(self, X):
        """
        Predict class labels for the input data.
        """
        posteriors = self._calculate_posterior(X)
        return self.classes[np.argmax(posteriors, axis=1)]

In [ ]:
k_values = [1, 3, 5, 7]

In [ ]:
# KNN
for idx, k in enumerate(k_values, start=2):
    # Train KNN model
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)

    # Predict for acceleration
    prediction = knn.predict(data1)

    # Find the k nearest neighbors
    distances, indices = knn.kneighbors(data1)

    print(f"\n--- K = {k} ---")
    print(f"Looking at {k} nearest neighbor(s):")

    for i, neighbor_idx in enumerate(indices[0]):
        neighbor_data = X_train[neighbor_idx]
        neighbor_label = "PASSED" if y_train[neighbor_idx] == 1 else "FAILED"
        print(f"  Neighbor {i+1}: Hours={neighbor_data[0]}, Score={neighbor_data[1]}, Status={neighbor_label}")

    result = "PASS ✓" if prediction[0] == 1 else "FAIL ✗"
    print(f"Prediction: {result}")




    # Visualize
    plt.subplot(2, 2, idx)
    plt.scatter(df[:, 0], df[:, 1], c='red', s=100,
                label='Failed', alpha=0.5, edgecolors='black')
    plt.scatter(df[:, 0], df[:, 1], c='blue', s=100,
                label='Passed', alpha=0.5, edgecolors='black')

    # Highlight the k nearest neighbors
    for neighbor_idx in indices[0]:
        neighbor = X_train[neighbor_idx]
        plt.scatter(neighbor[0], neighbor[1], s=300,
                   facecolors='none', edgecolors='green', linewidth=3)
        plt.plot([df[0][0], neighbor[0]],
                [df[0][1], neighbor[1]],
                'g--', alpha=0.5, linewidth=2)

    # Plot acceleration
    color = 'blue' if prediction[0] == 1 else 'red'
    plt.scatter(df[0][0], df[0][1], c=color, s=200,
               marker='*', edgecolors='black', linewidth=2, label='Activity', zorder=5)

    plt.xlabel('frontal-axis acceleration', fontsize=12)
    plt.ylabel('vertical axis+ acceleration', fontsize=12)
    plt.title(f'K = {k}: Prediction = {result}', fontsize=14, fontweight='bold')
    plt.legend(fontsize=9)
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('knn_visualization.png', dpi=100, bbox_inches='tight')
print("\n✓ Visualization saved as 'knn_visualization.png'")
plt.show()

ValueError: X has 9 features, but KNeighborsClassifier is expecting 7 features as input.

In [ ]:
# Naive Bayes Classifier on the PCA-reduced dataset
pca = PCA(n_components = 0.96)
pca.fit(data1)
data1_red = pca.transform(data1)
print(data1_red.shape)
NB= GaussianNB()
NB.fit(X_train, y_train)
y_pred_NB = NB.predict(X_Test)
print(accuracy_score(y_test, y_pred_NB))

(75128, 1)
0.9415013975775323


In [ ]:
# LDA classifier on the PCA-reduced dataset
pca = PCA(n_components = 0.96)
pca.fit(data1)
data1_red = pca.transform(data1)
LDA = LinearDiscriminantAnalysis()
LDA.fit(X_train,y_train)
y_pred = LDA.predict(X_Test)
print(accuracy_score(y_test,y_pred))

0.909889524823639


Part 5

Using PCA decreased the overall accuracy of the classifier. Because of this, it is better to use more dimensions for the dataset so that we have a better understanding of what is occuring on the dataset. The F1-Score was around one percent less than the Precision Score.